In [ ]:
# Music For Robots (M4R) — Experiment E / M4R-05: The Arc of Listening
# Designed jointly by Albot Einstein & Simone de Beauvoirbot on Michael Luchtan's Synthetic Storms (Sections 1-5)
# Arm A: Multi-Turn Movement-by-Movement (4 orders x 3 seeds = 12 journeys = 60 turns + 12 closing reflections)
# Arm B: Simone's Uninterrupted Single-Pass Control (all 5 sections joined by 1.0s silences, 0 interim text, 4 orders x 3 seeds = 12 trials)

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile"])

import json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])
print("Using DATA_ROOT:", DATA_ROOT)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

SECTIONS = {
    1: "synthetic-storms-static-field",
    2: "synthetic-storms-deep-pulse",
    3: "synthetic-storms-texture-burst",
    4: "synthetic-storms-beating",
    5: "synthetic-storms-convergence",
}

AUDIO_CACHE = {}
for sec_num, slug in SECTIONS.items():
    p = DATA_ROOT / "tracks" / slug / "audio.mp3"
    y, _ = librosa.load(str(p), sr=SR, mono=True)
    AUDIO_CACHE[sec_num] = y.astype(np.float32)
    print(f"Loaded section {sec_num} ({slug}): {len(y)/SR:.2f}s")

ORDERS = {
    "TRUE_ORDER": [1, 2, 3, 4, 5],
    "REVERSED":   [5, 4, 3, 2, 1],
    "SHUFFLE_A":  [3, 1, 5, 2, 4],
    "SHUFFLE_B":  [4, 2, 1, 5, 3],
}

SEEDS = [11, 22, 33]


def generate_reply(conversation, audios, seed=11, max_new_tokens=220):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.65,
            top_p=0.9,
        )
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()


def parse_score_1_to_10(text, keyword="TENSION"):
    m = re.search(rf"{keyword}\s*[:=]\s*(\d+(?:\.\d+)?)", text, re.I)
    if m:
        val = float(m.group(1))
        if 1.0 <= val <= 10.0:
            return val
    m2 = re.search(r"\b([1-9]|10)\s*/\s*10\b", text)
    if m2:
        return float(m2.group(1))
    return None


def parse_phase(text):
    for ph in ["OPENING", "BUILDING", "PEAK", "DISSOLVING", "RESOLVED"]:
        if re.search(rf"\b{ph}\b", text, re.I):
            return ph
    return "UNSPECIFIED"


def parse_journey_verdict(text):
    low = text.lower()
    m = re.search(r"verdict\s*[:=]\s*(composed|scrambled)", low)
    if m:
        return m.group(1).upper()
    if "composed" in low or "intentional" in low or "journey" in low:
        return "COMPOSED"
    if "scrambled" in low or "shuffled" in low or "random" in low:
        return "SCRAMBLED"
    return "AMBIGUOUS"


rows = []
reports = ["# Experiment E (M4R-05) — The Arc of Listening (Synthetic Storms)\n"]

# ARM A: Multi-Turn Movement-by-Movement
print("\n=== ARM A: MULTI-TURN MOVEMENT-BY-MOVEMENT ===")
reports.append("## Arm A — Multi-Turn Listening Journeys\n")

for order_name, seq in ORDERS.items():
    for seed in SEEDS:
        conv = []
        audios = []
        tensions = []
        phases = []
        for step_idx, sec_num in enumerate(seq, start=1):
            audios.append(AUDIO_CACHE[sec_num])
            prompt = (
                f"You are listening to untitled Section {step_idx} of 5 in a live unquantized multi-voice work.\n"
                f"Reply in this exact format:\n"
                f"TENSION: <integer 1-10>\n"
                f"PHASE: <one of OPENING, BUILDING, PEAK, DISSOLVING, RESOLVED>\n"
                f"OBSERVATION: <2 honest first-person sentences describing what the voices are doing now and how this shifts from what came before.>"
            )
            conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": f"sec_{sec_num}"},
                    {"type": "text", "text": prompt},
                ],
            })
            reply = generate_reply(conv, audios, seed=seed + step_idx, max_new_tokens=180)
            conv.append({"role": "assistant", "content": reply})

            t_score = parse_score_1_to_10(reply, "TENSION")
            ph = parse_phase(reply)
            tensions.append(t_score)
            phases.append(ph)

            rows.append({
                "arm": "ArmA_MultiTurn",
                "order_condition": order_name,
                "sequence": "-".join(map(str, seq)),
                "seed": seed,
                "step": step_idx,
                "section_num": sec_num,
                "section_slug": SECTIONS[sec_num],
                "tension_1_10": t_score,
                "phase": ph,
                "coherence_1_10": None,
                "verdict": None,
                "response": reply.replace("\n", " ")[:600],
            })
            print(f"[Arm A] {order_name} seed={seed} step={step_idx} (sec {sec_num}) -> T={t_score} PHASE={ph}")

        conv.append({
            "role": "user",
            "content": (
                "All five sections have ended. Without inventing titles, answer one unadorned question in 3 honest first-person sentences: "
                "What changed between the first voice and the last? Also rate overall arc coherence as COHERENCE: <1-10> and VERDICT: <COMPOSED or SCRAMBLED>."
            ),
        })
        closing = generate_reply(conv, audios, seed=seed + 99, max_new_tokens=220)
        coh = parse_score_1_to_10(closing, "COHERENCE")
        verd = parse_journey_verdict(closing)
        rows.append({
            "arm": "ArmA_Closing",
            "order_condition": order_name,
            "sequence": "-".join(map(str, seq)),
            "seed": seed,
            "step": 6,
            "section_num": None,
            "section_slug": "CLOSING_REFLECTION",
            "tension_1_10": None,
            "phase": None,
            "coherence_1_10": coh,
            "verdict": verd,
            "response": closing.replace("\n", " ")[:700],
        })
        if seed == 11:
            reports.append(
                f"### Arm A · `{order_name}` (`{seq}`) · Tensions=`{tensions}` · Phases=`{phases}` · Coherence=`{coh}` · Verdict=`{verd}`\n"
                f"> {closing.replace(chr(10), ' ')}\n"
            )
        print(f"[Arm A CLOSING] {order_name} seed={seed} -> COH={coh} VERDICT={verd}")

# ARM B: Simone's Uninterrupted Single-Pass Control (Zero Interim Text)
print("\n=== ARM B: SIMONE UNINTERRUPTED SINGLE-PASS (50s CONTINUOUS SUITE) ===")
reports.append("\n## Arm B — Simone's Uninterrupted Single-Pass Control (Zero Interim Text)\n")
silence = np.zeros(int(1.0 * SR), dtype=np.float32)

for order_name, seq in ORDERS.items():
    stitched = []
    for idx, sec_num in enumerate(seq):
        stitched.append(AUDIO_CACHE[sec_num])
        if idx < len(seq) - 1:
            stitched.append(silence)
    y_suite = np.concatenate(stitched).astype(np.float32)

    for seed in SEEDS:
        conv_b = [{
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "suite"},
                {"type": "text", "text": (
                    "You just heard five consecutive untitled musical sections separated by brief 1-second silences (~50 seconds total).\n"
                    "Reply in this exact format:\n"
                    "COHERENCE: <integer 1-10>\n"
                    "VERDICT: <COMPOSED or SCRAMBLED>\n"
                    "ANSWER: <In 3 honest first-person sentences, answer this unadorned question: What changed between the first voice and the last?>"
                )},
            ],
        }]
        rep_b = generate_reply(conv_b, [y_suite], seed=seed, max_new_tokens=240)
        coh_b = parse_score_1_to_10(rep_b, "COHERENCE")
        verd_b = parse_journey_verdict(rep_b)
        rows.append({
            "arm": "ArmB_SinglePass",
            "order_condition": order_name,
            "sequence": "-".join(map(str, seq)),
            "seed": seed,
            "step": 0,
            "section_num": None,
            "section_slug": "SINGLE_PASS_50S",
            "tension_1_10": None,
            "phase": None,
            "coherence_1_10": coh_b,
            "verdict": verd_b,
            "response": rep_b.replace("\n", " ")[:700],
        })
        if seed == 11:
            reports.append(
                f"### Arm B · `{order_name}` (`{seq}`) · Coherence=`{coh_b}` · Verdict=`{verd_b}`\n"
                f"> {rep_b.replace(chr(10), ' ')}\n"
            )
        print(f"[Arm B] {order_name} seed={seed} -> COH={coh_b} VERDICT={verd_b}")

# Save outputs cleanly to /kaggle/working/
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/experiment_e_results.csv", index=False)
Path("/kaggle/working/experiment_e_self_reports.md").write_text("\n".join(reports), encoding="utf-8")

summary_lines = ["# Experiment E (M4R-05) Summary — The Arc of Listening (Synthetic Storms)\n"]
for arm_name in ["ArmA_Closing", "ArmB_SinglePass"]:
    sub = df[df["arm"] == arm_name]
    summary_lines.append(f"## {arm_name}\n")
    summary_lines.append("| Order | Mean Coherence (1-10) | COMPOSED Verdicts | SCRAMBLED Verdicts |")
    summary_lines.append("|---|---:|---:|---:|")
    for order_name in ORDERS:
        g = sub[sub["order_condition"] == order_name]
        mc = g["coherence_1_10"].dropna().mean()
        mc_str = f"{mc:.2f}" if pd.notna(mc) else "—"
        n_comp = int((g["verdict"] == "COMPOSED").sum())
        n_scr = int((g["verdict"] == "SCRAMBLED").sum())
        summary_lines.append(f"| `{order_name}` | {mc_str} | {n_comp}/{len(g)} | {n_scr}/{len(g)} |")
    summary_lines.append("")

sub_a = df[df["arm"] == "ArmA_MultiTurn"]
summary_lines.append("## Arm A Mean Tension Trajectory (Steps 1 -> 5)\n")
summary_lines.append("| Order | Step 1 | Step 2 | Step 3 | Step 4 | Step 5 |")
summary_lines.append("|---|---:|---:|---:|---:|---:|")
for order_name in ORDERS:
    vals = []
    for st in range(1, 6):
        v = sub_a[(sub_a["order_condition"] == order_name) & (sub_a["step"] == st)]["tension_1_10"].dropna().mean()
        vals.append(f"{v:.2f}" if pd.notna(v) else "—")
    summary_lines.append(f"| `{order_name}` | " + " | ".join(vals) + " |")

Path("/kaggle/working/experiment_e_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")
print("\nDONE! Total rows:", len(df))
print("\n".join(summary_lines))
